## Music and Texts

In [13]:
import os
import json
import requests
from bs4 import BeautifulSoup

# Constants
BASE_URL = "https://nationalanthems.info/"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.93 Safari/537.36'
}

def load_country_data(filename):
    """Loads country data from a JSON file."""
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            return json.load(file)
    except Exception as e:
        print(f"Failed to load country data: {e}")
        return None

def select_countries(countries, num_countries=100):
    """Selects a specified number of countries from the list."""
    country_keys = list(countries.keys())
    country_keys.sort()  # Sort alphabetically
    return {code: countries[code] for code in country_keys[:num_countries]}

def fetch_country_page(code):
    """Fetches a country's page and extracts the English translation and MP3 link."""
    url = f"{BASE_URL}{code.lower()}.htm"
    try:
        response = requests.get(url, headers=HEADERS)
    except Exception as e:
        print(f"Error fetching URL {url}: {e}")
        return None, None
    
    if response.status_code != 200:
        print(f"Failed to retrieve page for {code} at {url}. Status code: {response.status_code}")
        return None, None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Extract English translation
    translation_div = soup.find("div", class_="collapseomatic", title=lambda t: t in ["English translation", "English lyrics"])
    if translation_div:
        content_id = f"target-{translation_div['id']}"
        content_div = soup.find("div", id=content_id)
        if content_div:
            translation_text = content_div.get_text(separator=" ", strip=True)
            translation = translation_text
        else:
            print("Content div not found.")
            translation = None
    else:
        print("Header div with title 'English translation' not found.")
        translation = None
    
    # Construct MP3 link
    mp3_link = f"{BASE_URL}{code.lower()}.mp3"
    
    return translation, mp3_link

def save_text_to_file(content, filepath):
    """Saves text content to a file."""
    try:
        with open(filepath, 'w', encoding='utf-8') as file:
            file.write(content)
    except Exception as e:
        print(f"Error saving text to {filepath}: {e}")

def download_file_from_url(url, filepath):
    """Downloads a file from a URL and saves it to a specified path."""
    try:
        response = requests.get(url, headers=HEADERS, stream=True)
        if response.status_code == 200:
            with open(filepath, 'wb') as file:
                for chunk in response.iter_content(chunk_size=1024):
                    if chunk:
                        file.write(chunk)
        else:
            print(f"Failed to download file from {url}. Status code: {response.status_code}")
    except Exception as e:
        print(f"Error downloading {url}: {e}")

def main():
    # Load country data
    countries = load_country_data("countries_metadata.json")
    
    if countries is None:
        return
    
    # Select countries
    selected_countries = select_countries(countries, num_countries=100)
    print(selected_countries)
    print(f"Selected {len(selected_countries)} countries for processing.")
    
    # Create directories
    os.makedirs("translations", exist_ok=True)
    os.makedirs("mp3", exist_ok=True)
    
    for code, name in selected_countries.items():
        print(f"\nProcessing {code} - {name}")
        try:
            translation, mp3_link = fetch_country_page(code)
        except Exception as e:
            print(f"Error processing {code} - {name}: {e}. Skipping.")
            continue
        
        if not translation:
            print(f"No English translation found for {name}. Skipping MP3 download.")
            continue
        
        # Save translation
        safe_name = name.replace(" ", "_")
        translation_filepath = os.path.join("translations", f"{code}_{safe_name}.txt")
        try:
            save_text_to_file(translation, translation_filepath)
            print(f"Saved translation to {translation_filepath}")
        except Exception as e:
            print(f"Error saving translation for {name}: {e}. Skipping.")
            continue
        
        # Download MP3
        if mp3_link:
            mp3_filepath = os.path.join("mp3", f"{code}_{safe_name}.mp3")
            try:
                download_file_from_url(mp3_link, mp3_filepath)
                print(f"Downloaded MP3 to {mp3_filepath}")
            except Exception as e:
                print(f"Error downloading MP3 for {name}: {e}. Skipping MP3.")
        else:
            print(f"No MP3 file found for {name}.")

if __name__ == "__main__":
    main()


{'AD': 'Andorra', 'AE': 'United Arab Emirates', 'AF': 'Afghanistan', 'AG': 'Antigua and Barbuda', 'AI': 'Anguilla', 'AL': 'Albania', 'AM': 'Armenia', 'AO': 'Angola', 'AQ': 'Antarctica', 'AR': 'Argentina', 'AS': 'American Samoa', 'AT': 'Austria', 'AU': 'Australia', 'AW': 'Aruba', 'AX': 'Åland Islands', 'AZ': 'Azerbaijan', 'BA': 'Bosnia and Herzegovina', 'BB': 'Barbados', 'BD': 'Bangladesh', 'BE': 'Belgium', 'BF': 'Burkina Faso', 'BG': 'Bulgaria', 'BH': 'Bahrain', 'BI': 'Burundi', 'BJ': 'Benin', 'BL': 'Saint Barthélemy', 'BM': 'Bermuda', 'BN': 'Brunei Darussalam', 'BO': 'Bolivia, Plurinational State of', 'BQ': 'Caribbean Netherlands', 'BR': 'Brazil', 'BS': 'Bahamas', 'BT': 'Bhutan', 'BV': 'Bouvet Island', 'BW': 'Botswana', 'BY': 'Belarus', 'BZ': 'Belize', 'CA': 'Canada', 'CC': 'Cocos (Keeling) Islands', 'CD': 'Congo, the Democratic Republic of the', 'CF': 'Central African Republic', 'CG': 'Republic of the Congo', 'CH': 'Switzerland', 'CI': "Côte d'Ivoire", 'CK': 'Cook Islands', 'CL': 'Ch

## Flags

In [15]:
import os
import requests

# Set headers to mimic a browser
headers = {"User-Agent": "Mozilla/5.0"}

# API URL for the "svg" folder of the repository on the main branch
api_url = "https://api.github.com/repos/hampusborgos/country-flags/contents/svg?ref=main"

response = requests.get(api_url, headers=headers)
if response.status_code != 200:
    raise Exception(f"Failed to fetch file list: {response.status_code}")

files = response.json()

# Sort files by name for a consistent order
files_sorted = sorted(files, key=lambda x: x['name'])

# Create an output directory for the flags
output_dir = "flags"
os.makedirs(output_dir, exist_ok=True)

# Download every flag (SVG format)
for file_info in files_sorted:
    download_url = file_info["download_url"]
    file_name = file_info["name"]  # e.g., "ad.svg"
    r = requests.get(download_url, headers=headers)
    if r.status_code == 200:
        with open(os.path.join(output_dir, file_name), "wb") as f:
            f.write(r.content)
        print(f"Downloaded {file_name}")
    else:
        print(f"Failed to download {file_name} (status code: {r.status_code})")

Downloaded ad.svg
Downloaded ae.svg
Downloaded af.svg
Downloaded ag.svg
Downloaded ai.svg
Downloaded al.svg
Downloaded am.svg
Downloaded ao.svg
Downloaded aq.svg
Downloaded ar.svg
Downloaded as.svg
Downloaded at.svg
Downloaded au.svg
Downloaded aw.svg
Downloaded ax.svg
Downloaded az.svg
Downloaded ba.svg
Downloaded bb.svg
Downloaded bd.svg
Downloaded be.svg
Downloaded bf.svg
Downloaded bg.svg
Downloaded bh.svg
Downloaded bi.svg
Downloaded bj.svg
Downloaded bl.svg
Downloaded bm.svg
Downloaded bn.svg
Downloaded bo.svg
Downloaded bq.svg
Downloaded br.svg
Downloaded bs.svg
Downloaded bt.svg
Downloaded bv.svg
Downloaded bw.svg
Downloaded by.svg
Downloaded bz.svg
Downloaded ca.svg
Downloaded cc.svg
Downloaded cd.svg
Downloaded cf.svg
Downloaded cg.svg
Downloaded ch.svg
Downloaded ci.svg
Downloaded ck.svg
Downloaded cl.svg
Downloaded cm.svg
Downloaded cn.svg
Downloaded co.svg
Downloaded cr.svg
Downloaded cu.svg
Downloaded cv.svg
Downloaded cw.svg
Downloaded cx.svg
Downloaded cy.svg
Downloaded